In [10]:
import numpy as np
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
tf.random.set_seed(42)

X, y = make_moons(
    n_samples=1200,
    noise=0.30,
    random_state=42
)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(960, 2)
(240, 2)


In [12]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),

    tf.keras.layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),

    tf.keras.layers.Dropout(0.30),

    tf.keras.layers.Dense(
        16,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),

    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 32)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 641 (2.50 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    min_delta=0.0001,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

print("\nEpochs actually completed : ", len(history.history['loss']))

Epoch 1/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5326 - loss: 0.7098 - val_accuracy: 0.8021 - val_loss: 0.6491
Epoch 2/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 902us/step - accuracy: 0.7839 - loss: 0.6089 - val_accuracy: 0.8021 - val_loss: 0.5718
Epoch 3/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 827us/step - accuracy: 0.8151 - loss: 0.5342 - val_accuracy: 0.8073 - val_loss: 0.5103
Epoch 4/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 824us/step - accuracy: 0.8464 - loss: 0.4755 - val_accuracy: 0.8177 - val_loss: 0.4660
Epoch 5/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 867us/step - accuracy: 0.8581 - loss: 0.4185 - val_accuracy: 0.8125 - val_loss: 0.4365
Epoch 6/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 852us/step - accuracy: 0.8503 - loss: 0.4049 - val_accuracy: 0.8125 - val_loss: 0.4229
Epoch 7/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 836us/step - accuracy: 0.8581 - loss: 0.3714 - val_accuracy: 0.8125 - val_loss: 0.4182
Epoch 8/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 784us/step - accuracy: 0.8581 - loss: 0.3878 - val_accura

In [14]:
test_loss, test_accuracy = model.evaluate(
    X_test, y_test, verbose=0
)

print("\nTest loss : ", round(test_loss, 4))
print("\nTest accuracy : ", round(test_accuracy, 4))

probabilities = model.predict(X_test, verbose=0).ravel()

predictions = (probabilities >= 0.5).astype(int)

print("\nFirst 10 Probabilities")
print(np.round(probabilities[:10], 3))

print("\nFirst 10 Predicted Calsses")
print(predictions[:10])

print("Classification Report")
print(classification_report(y_test, predictions))


Test loss :  0.2466

Test accuracy :  0.9125

First 10 Probabilities
[0.995 0.991 0.006 0.023 0.017 0.004 0.928 0.963 0.5   0.319]

First 10 Predicted Calsses
[1 1 0 0 0 0 1 1 1 0]
Classification Report
              precision    recall  f1-score   support

           0       0.90      0.93      0.91       120
           1       0.92      0.90      0.91       120

    accuracy                           0.91       240
   macro avg       0.91      0.91      0.91       240
weighted avg       0.91      0.91      0.91       240

